# Chapter 02: 3D Camera Rig & Inverse Perspective Mapping (IPM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvknyn/self-driving-ai-course/blob/main/notebooks/02_camera_geometry_and_ipm.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717.svg)](https://github.com/vvknyn/self-driving-ai-course)

> **The Big Question**: *How do we unproject 2D camera pixels onto a metric 3D ground plane—and why does a 1.5° chassis tilt cause 40% distance error?*

---

## 1. 🚨 The Real-World Dilemma
Perspective cameras divide by depth $Z_c$. Under the Flat Ground Assumption ($Z=0$), we can construct a Planar Homography $H = K [r_1, r_2, \mathbf{t}]$. But suspension bounce changes pitch angle, causing lanes to violently flare outward into hyperbolas!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def compute_intrinsics(width=1920, height=1080, hfov_deg=90.0):
    hfov_rad = np.deg2rad(hfov_deg)
    fx = (width / 2.0) / np.tan(hfov_rad / 2.0)
    return np.array([
        [fx, 0.0, width / 2.0],
        [0.0, fx, height / 2.0],
        [0.0, 0.0, 1.0]
    ])

def euler_to_rotation(roll, pitch, yaw):
    Rx = np.array([[1, 0, 0], [0, np.cos(roll), -np.sin(roll)], [0, np.sin(roll), np.cos(roll)]])
    Ry = np.array([[np.cos(pitch), 0, np.sin(pitch)], [0, 1, 0], [-np.sin(pitch), 0, np.cos(pitch)]])
    Rz = np.array([[np.cos(yaw), -np.sin(yaw), 0], [np.sin(yaw), np.cos(yaw), 0], [0, 0, 1]])
    return Rz @ Ry @ Rx

def make_homography(K, R, t):
    Rt = np.hstack([R[:, 0:1], R[:, 1:2], t.reshape(3, 1)])
    H = K @ Rt
    return H / H[2, 2]

K = compute_intrinsics()
R_nominal = euler_to_rotation(0, np.deg2rad(3.0), 0)
t_nominal = np.array([0.0, -1.4, 0.0])
H_nominal = make_homography(K, R_nominal, t_nominal)
print("Nominal Planar Homography Matrix H:")
print(np.round(H_nominal, 4))

In [ ]:
# Pitch error simulation during hard braking (+2 degrees)
R_perturbed = euler_to_rotation(0, np.deg2rad(5.0), 0)
H_perturbed = make_homography(K, R_perturbed, t_nominal)
H_inv_nominal = np.linalg.inv(H_nominal)

y_pts = np.linspace(5.0, 45.0, 50)
left_line = np.stack([-1.85 * np.ones_like(y_pts), y_pts, np.ones_like(y_pts)])
right_line = np.stack([1.85 * np.ones_like(y_pts), y_pts, np.ones_like(y_pts)])

def warp_line(line, H_fwd, H_bwd):
    p_img = H_fwd @ line
    u = p_img[0] / p_img[2]
    v = p_img[1] / p_img[2]
    p_bev = H_bwd @ np.stack([u, v, np.ones_like(u)])
    return p_bev[0] / p_bev[2], p_bev[1] / p_bev[2]

xl_err, yl_err = warp_line(left_line, H_perturbed, H_inv_nominal)
xr_err, yr_err = warp_line(right_line, H_perturbed, H_inv_nominal)

plt.figure(figsize=(8, 4))
plt.plot([-1.85, -1.85], [5, 45], 'g--', label="True Parallel Lane")
plt.plot([1.85, 1.85], [5, 45], 'g--')
plt.plot(xl_err, yl_err, 'r-', lw=2, label="Reconstructed with +2° Pitch Error")
plt.plot(xr_err, yr_err, 'r-', lw=2)
plt.title("The Pitch Flare Effect in IPM Reconstruction")
plt.xlabel("Lateral X (m)")
plt.ylabel("Longitudinal Y (m)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. 🩺 Andrew Ng's Diagnostic Field Guide

| Observed Symptom | Underlying Mathematical Mechanism | Verification Test | Production Fix |
| :--- | :--- | :--- | :--- |
| **Lanes flare outward at distance** | Pitch over-estimation; camera unprojects rays into ground too early. | Check if distant lane width $> 3.7\text{m}$. | Recalibrate pitch extrinsic down. |
| **Distance errors oscillate during braking** | Dynamic chassis suspension bounce ($\Delta \theta \approx 2^\circ$). | Correlate distance error with longitudinal accelerometer. | Ingest live IMU pitch rate into dynamic $H(t)$. |